# Lecture 1-3 exercise
> This exercise is not graded, but it can help you to get familiar with the lecture contents, how to setup the environment.

## Setup your local environment

### Python (pygismo)

Clone the course repository and install the dependencies:

```bash
git clone https://github.com/Crazy-Rich-Meghan/WI4450-spline-demo.git
cd WI4450-spline-demo
conda create -n gismo-slides python=3.10 -y
conda activate gismo-slides
pip install -r requirements.txt
pip install ipykernel
python -m ipykernel install --user --name gismo-slides --display-name "Python (gismo-slides)"
```
Then select the **Python (gismo-slides)** kernel in Jupyter before running the notebook.

### C++ (G+Smo library)

Please see the full documentation: [gismo.github.io](https://gismo.github.io/)

> **Note on NURBS in pygismo.** The current pygismo (25.7.0) exposes B-spline classes (`gsBSpline`, `gsTensorBSpline2`) but not the NURBS classes (`gsNurbs`, `gsTensorNurbs`) — the C++ code exists, but the Python bindings are not yet wired up.

In [ ]:
# Important! Please read such that you know where to find functions.
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import pygismo as gs
# Check available modules in pygismo
print([x for x in dir(gs) if not x.startswith('_')])
# List all functions in the gs.nurbs module
print([x for x in dir(gs.nurbs) if x.startswith('gs')])
# Cjeck the documentation for gs.nurbs.gsBSpline
help(gs.nurbs.gsBSpline)
# Check the documentation for gs.pde.gsBoundaryConditions
help(gs.pde.gsBoundaryConditions)

# Use https://gismo.github.io/ to see full documentations

plt.rcParams.update({'figure.figsize': (11, 7), 'font.size': 12, 'axes.grid': True})




In [ ]:
# Helper functions

# For ploting vts file with nurbs
def write_nurbs_surface_vts(filename, projected_surf, n_u=50, n_v=80):
    u_vals = np.linspace(0, 1, n_u)
    v_vals = np.linspace(0, 1, n_v)
    uv = np.vstack(np.meshgrid(u_vals, v_vals, indexing="ij")).reshape(2, -1)
    raw_eval = np.asarray(projected_surf.eval(uv))
    physical = raw_eval[:3, :] / raw_eval[3, :]
    points = physical.T.reshape(n_u, n_v, 3)
    with open(filename, "w") as f:
        f.write('<?xml version="1.0"?>\n')
        f.write('<VTKFile type="StructuredGrid" version="0.1">\n')
        f.write(f'<StructuredGrid WholeExtent="0 {n_u-1} 0 {n_v-1} 0 0">\n')
        f.write(f'<Piece Extent="0 {n_u-1} 0 {n_v-1} 0 0">\n')
        f.write('<Points>\n')
        f.write('<DataArray type="Float64" NumberOfComponents="3" format="ascii">\n')
        for v in range(n_v):
            for u in range(n_u):
                x, y, z = points[u, v]
                f.write(f"{x:.10f} {y:.10f} {z:.10f}\n")
        f.write('</DataArray>\n</Points>\n')
        f.write('</Piece>\n</StructuredGrid>\n</VTKFile>\n')

# Evaluate NURBS in projective space
def eval_rational(curve, u_vals):
    samples = []
    for u in np.atleast_1d(u_vals):
        samples.append(np.asarray(curve.eval([u])).flatten())
    raw = np.vstack(samples)
    w = raw[:, -1][:, None]
    return raw[:, :-1] / w

### Exercise 1: NURBS mechanical part cross-section

Model both the outer boundary and the inner circle.

![Engineering drawing cross-section](figure/engineering_drawing-1.png)


*Figure: Engineering drawing*


#### Part 1: Identify the segments

Trace the outer boundary counterclockwise, note where the curve type changes, and list all seven segments with their types and endpoints.


#### Part 2: Control points and weights

Each degree-2 segment uses three control points; lines use endpoints and midpoints (weights $1$), while circular/elliptical arcs insert the tangent intersection as the middle CP (weights $1, w, 1$ with $w = \frac{\sqrt{2}}{2}$). 


#### Part 3: Merge into a single NURBS curve


#### Part 4: Inner circle


In [ ]:
# Draft program to start with.
w = 1.0 / np.sqrt(2.0)

outer_control_points = np.array([
    # TODO
], dtype=float).reshape(15, 2)

outer_weights = np.array([
    # TODO
])

outer_knots = [
    # TODO
]

inner_control_points = np.array([
    # TODO
], dtype=float).reshape(9, 2)

inner_weights = np.array([
    # TODO
])

inner_knots = [
    # TODO
]

outer_curve = gs.nurbs.gsBSpline(
    gs.nurbs.gsKnotVector(outer_knots, 2),
    np.hstack([outer_control_points * outer_weights[:, None], outer_weights[:, None]])
)

inner_curve = gs.nurbs.gsBSpline(
    gs.nurbs.gsKnotVector(inner_knots, 2),
    np.hstack([inner_control_points * inner_weights[:, None], inner_weights[:, None]])
)

# Evaluate with gsBSpline.eval([...]) and de-project using the last column. Need to call eval_rational to do this. 

### Exercise 2: Swept surface construction

Complete the trajectory, profile, and sweeping steps. Make sure the profile circles remain perpendicular to the trajectory so the swept cross-section stays circular.

![Sketch of the swept surface](figure/sketch_swept_surface.png)

#### Part 1: Trajectory curve

Set the control points for a quadratic trajectory in the $xz$-plane. This define the path used for the sweep.


#### Part 2: Profile circle

This is the same as Exercise 1. Pick a radius as you wish.


#### Part 3: Swept surface grid

Assemble the control net by placing the circle profile at each trajectory control point, keeping the profile perpendicular to the local tangent. The homogeneous control points come from multiplying each profile CP by its weight and the current trajectory weight.


#### Part 4: Evaluation and export

Once the control grid is ready, evaluate the projective surface. And plot by using the helper function write_nurbs_surface_vts


In [ ]:
trajectory_control_points = np.zeros((3, 3))  # TODO: fill control points
trajectory_knots = []  # TODO

profile_control_points = np.zeros((9, 2))  # TODO: circle control points
profile_weights = np.ones(9)  # TODO
profile_knots = []  # TODO

w_circle = 1.0 / np.sqrt(2.0)

# q^1(u) = (cx, cy) in the local XY plane; M(v) is the frame [local_x, local_y, tangent].

# p(u,v)=q^2(v)+M(v)q^1(u)

